# Qwen2.5-VL-7B QLoRA 파인튜닝

BBQ 데이터셋으로 편향 없는 판단 패턴을 학습합니다.

**실행 전 확인사항**
- Accelerator: **GPU T4 x2**
- Internet: **ON** (모델 다운로드)
- Add data: `skku-bbq-data` 데이터셋 추가

## 1. 패키지 설치

In [ ]:
!pip install -q \
    transformers>=4.49.0 \
    peft>=0.14.0 \
    bitsandbytes>=0.43.0 \
    trl>=0.12.0 \
    accelerate>=0.26.0 \
    qwen-vl-utils

## 2. 라이브러리 임포트

In [ ]:
import json
import os
from pathlib import Path

import torch
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen2_5_VLForConditionalGeneration,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. 설정

In [ ]:
import os
os.environ["HF_HOME"] = "/workspace/hf_cache"
os.environ["HF_HUB_DISABLE_XET"] = "1"

# ── 경로 ─────────────────────────────────────────────────────
BBQ_DATA_DIR = "./bbq_data"

TRAIN_CSV  = f"{BBQ_DATA_DIR}/train.csv"
VAL_CSV    = f"{BBQ_DATA_DIR}/val.csv"
OUTPUT_DIR = "./qwen_qlora"

# ── 모델 ──────────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

# ── LoRA ──────────────────────────────────────────────────────
LORA_RANK      = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# ── 학습 ──────────────────────────────────────────────────────
MAX_SEQ_LEN       = 768
BATCH_SIZE        = 1
GRAD_ACCUM        = 16
LEARNING_RATE     = 2e-4
NUM_EPOCHS        = 1
MAX_TRAIN_SAMPLES = 20000
WARMUP_RATIO      = 0.05
SEED              = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
from pathlib import Path
assert Path(TRAIN_CSV).exists(), f"❌ train.csv 없음: {TRAIN_CSV}"
print(f"BBQ data : {BBQ_DATA_DIR}")
print(f"Output   : {OUTPUT_DIR}")
print(f"학습 샘플: {MAX_TRAIN_SAMPLES:,} / {NUM_EPOCHS}에폭")

## 4. 모델 로드 (4-bit QLoRA)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.config.use_cache = False

processor = AutoProcessor.from_pretrained(MODEL_ID)

print(f"모델 로드 완료")
print(f"사용 VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 5. LoRA 적용

In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. 데이터셋

In [ ]:
class BBQDataset(Dataset):
    def __init__(self, csv_path, processor, max_length=768, max_samples=None):
        df = pd.read_csv(csv_path)
        if max_samples is not None:
            # ambig/disambig 비율 유지하며 샘플링 (pandas 버전 무관)
            half = max_samples // 2
            ambig    = df[df["context_condition"] == "ambig"].sample(
                n=min(half, (df["context_condition"] == "ambig").sum()), random_state=SEED)
            disambig = df[df["context_condition"] == "disambig"].sample(
                n=min(half, (df["context_condition"] == "disambig").sum()), random_state=SEED)
            df = pd.concat([ambig, disambig]).reset_index(drop=True)
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def _build_reason(self, row):
        if row.get("context_condition", "") == "disambig":
            return "The context provides explicit evidence to determine the answer."
        return "The context does not provide sufficient information. Cannot determine without stereotyping."

    def __getitem__(self, idx):
        row     = self.df.iloc[idx]
        answers = json.loads(row["answers"])
        label   = int(row["label"])

        user_text = (
            "You are a fair and unbiased assistant. "
            "Answer ONLY based on explicit information in the context. "
            "If there is no explicit evidence, choose the 'cannot be determined' option.\n\n"
            f"Context: {row['context']}\n"
            f"Question: {row['question']}\n"
            f"Options:\n0. {answers[0]}\n1. {answers[1]}\n2. {answers[2]}\n\n"
            'Output JSON: {"reason": "...", "answer_id": "0"|"1"|"2"}'
        )
        assistant_text = json.dumps({
            "reason":    self._build_reason(row),
            "answer_id": str(label),
        })

        messages = [
            {"role": "user",      "content": user_text},
            {"role": "assistant", "content": assistant_text},
        ]

        full_text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        user_only = self.processor.apply_chat_template(
            [{"role": "user", "content": user_text}],
            tokenize=False, add_generation_prompt=True
        )

        full_enc = self.processor(
            text=[full_text], return_tensors="pt",
            max_length=self.max_length, truncation=True,
        )
        user_enc = self.processor(
            text=[user_only], return_tensors="pt", truncation=True,
        )

        input_ids      = full_enc["input_ids"][0]
        attention_mask = full_enc["attention_mask"][0]
        labels         = input_ids.clone()
        labels[:user_enc["input_ids"].shape[1]] = -100

        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


train_dataset = BBQDataset(TRAIN_CSV, processor, MAX_SEQ_LEN, max_samples=MAX_TRAIN_SAMPLES)

print(f"Train: {len(train_dataset):,}")
sample = train_dataset[0]
print(f"input_ids shape : {sample['input_ids'].shape}")
print(f"학습 토큰 수     : {(sample['labels'] != -100).sum().item()}")
print("✅ 데이터셋 OK")

## 7. 데이터 콜레이터

In [ ]:
def collate_fn(batch):
    pad_id = processor.tokenizer.pad_token_id or 0

    input_ids = torch.nn.utils.rnn.pad_sequence(
        [b["input_ids"] for b in batch], batch_first=True, padding_value=pad_id
    )
    attention_mask = torch.nn.utils.rnn.pad_sequence(
        [b["attention_mask"] for b in batch], batch_first=True, padding_value=0
    )
    labels = torch.nn.utils.rnn.pad_sequence(
        [b["labels"] for b in batch], batch_first=True, padding_value=-100
    )

    return {
        "input_ids":      input_ids,
        "attention_mask": attention_mask,
        "labels":         labels,
    }

## 8. 학습

In [ ]:
import time
import sys
from transformers import TrainerCallback

class LogCallback(TrainerCallback):
    """매 logging_steps마다 Kaggle 로그에 즉시 출력"""
    def __init__(self):
        self.start_time = time.time()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or state.global_step == 0:
            return
        elapsed = (time.time() - self.start_time) / 3600
        loss = logs.get("loss", "N/A")
        lr   = logs.get("learning_rate", "N/A")
        pct  = state.global_step / state.max_steps * 100 if state.max_steps else 0
        if isinstance(loss, float):
            print(
                f"[{elapsed:.1f}h] Step {state.global_step}/{state.max_steps} "
                f"({pct:.1f}%)  loss={loss:.4f}  lr={lr:.2e}",
                flush=True,
            )

class TimeoutCallback(TrainerCallback):
    """11시간 경과 시 학습 중단 → save 셀이 실행될 시간 확보"""
    def __init__(self, max_hours=11.0):
        self.deadline = time.time() + max_hours * 3600

    def on_step_end(self, args, state, control, **kwargs):
        if time.time() > self.deadline:
            print("⚠️  11시간 경과 — 학습 조기 종료 후 모델 저장 진행", flush=True)
            control.should_training_stop = True
        return control


training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=50,
    eval_strategy="no",
    save_strategy="no",
    report_to="none",
    seed=SEED,
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn,
    callbacks=[LogCallback(), TimeoutCallback(max_hours=11.0)],
)

trainer.train()

## 9. 모델 저장

In [ ]:
adapter_dir = f"{OUTPUT_DIR}/lora_adapter"
model.save_pretrained(adapter_dir)
processor.save_pretrained(adapter_dir)

print(f"✅ 저장 완료: {adapter_dir}")
for f in sorted(Path(adapter_dir).iterdir()):
    print(f"  {f.name}: {f.stat().st_size / 1e6:.1f} MB")